In [1]:
from IPython.display import HTML

HTML(
    '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">'
)

# <i class="fas fa-code-branch" style="color:#18ab4b"></i> SageMaker Pipeline — process -> train, locally

A `LocalPipelineSession` pipeline: a <i class="fas fa-cog" style="color:#18ab4b"></i> **ProcessingStep** (`prep.py`) generates the <i class="fas fa-database" style="color:#18ab4b"></i> credit data and a <i class="fas fa-user-graduate" style="color:#18ab4b"></i> **TrainingStep** fits the CatBoost model on it — wired together as on AWS, with inter-step artifacts passing through oblako's **S3Proxy**. Mirrors `batch_scoring/.../sagemaker_pipeline.py`.

Prereqs: `oblako up s3proxy`, Docker, `pip install 'oblako[sagemaker,notebook]'`.

## <i class="fas fa-database" style="color:#18ab4b"></i> Wire S3Proxy as the artifact store + <i class="fas fa-file-code" style="color:#6c63ff"></i> build the container

In [2]:
import os
import tempfile

from sagemaker.estimator import Estimator
from sagemaker.processing import ProcessingOutput, ScriptProcessor
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import LocalPipelineSession
from sagemaker.workflow.steps import ProcessingStep, TrainingStep

from oblako.services import S3ProxyService, SageMakerService

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")

# SageMaker passes step artifacts via S3 -> point it at S3Proxy (path-style, checksum-off)
cfg = os.path.join(tempfile.mkdtemp(), "aws-config")
with open(cfg, "w") as fh:
    fh.write(
        "[default]\nregion = us-east-1\nrequest_checksum_calculation = when_required\n"
        "s3 =\n    addressing_style = path\n"
    )
os.environ["AWS_ENDPOINT_URL_S3"] = "http://localhost:9000"
os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_CONFIG_FILE"] = cfg

IMAGE = "oblako-credit-risk:latest"
ROLE = "arn:aws:iam::000000000000:role/dummy"
BUCKET = "credit-risk-pipeline"

SageMakerService().build_image(path="container", tag=IMAGE)
s3 = S3ProxyService().get_client()
try:
    s3.create_bucket(Bucket=BUCKET)
except Exception:
    pass

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/deburky/Library/Application Support/sagemaker/config.yaml


Step 1/7 : FROM python:3.11-slim
 ---> a3ab0b966bc4
Step 2/7 : RUN apt-get update && apt-get install -y --no-install-recommends libgomp1     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> dac6d10a3439
Step 3/7 : RUN pip install --no-cache-dir catboost numpy pandas joblib
 ---> Using cache
 ---> 8432245a91ff
Step 4/7 : COPY src/ /opt/program/
 ---> Using cache
 ---> 66f061362b95
Step 5/7 : WORKDIR /opt/program
 ---> Using cache
 ---> b6c9560c6fd2
Step 6/7 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> e724b186fc1d
Step 7/7 : ENTRYPOINT ["python", "main.py"]
 ---> Using cache
 ---> 2f70f8506e14
Successfully built 2f70f8506e14
Successfully tagged oblako-credit-risk:latest


## <i class="fas fa-code-branch" style="color:#18ab4b"></i> Define and run the pipeline

In [3]:
session = LocalPipelineSession()
session.default_bucket = lambda: BUCKET

processor = ScriptProcessor(
    image_uri="python:3.11-slim",
    command=["python"],
    instance_type="local",
    instance_count=1,
    role=ROLE,
    sagemaker_session=session,
)
prep = ProcessingStep(
    name="GenerateData",
    processor=processor,
    code="prep.py",
    outputs=[ProcessingOutput(output_name="train", source="/opt/ml/processing/output")],
)

estimator = Estimator(
    image_uri=IMAGE,
    role=ROLE,
    instance_count=1,
    instance_type="local",
    sagemaker_session=session,
    hyperparameters={
        "iterations": "150",
        "depth": "6",
        "learning-rate": "0.1",
        "factor": "20",
        "offset": "600",
    },
)
train_step = TrainingStep(
    name="TrainCreditModel",
    estimator=estimator,
    inputs={
        "train": prep.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri
    },
)

pipeline = Pipeline(
    name="oblako-credit-risk", steps=[prep, train_step], sagemaker_session=session
)
pipeline.upsert(role_arn=ROLE)
execution = pipeline.start()
print("pipeline:", execution.status)
for step in execution.list_steps()["PipelineExecutionSteps"]:
    print(" ", step["StepName"], step["StepStatus"])

INFO:botocore.credentials:Found credentials in environment variables.


INFO:botocore.configprovider:Found endpoint for s3 via: environment_service.


INFO:botocore.configprovider:Found endpoint for s3 via: environment_service.


INFO:botocore.credentials:Found credentials in environment variables.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


...

qws46dxd6z-algo-1-zatuf  | trained CatBoost on 600 rows, 12 features (default rate 0.39)
qws46dxd6z-algo-1-zatuf exited with code 0
 Compose Stopping Aborting on container exit...
 Container qws46dxd6z-algo-1-zatuf Stopping 
 Container qws46dxd6z-algo-1-zatuf Stopped 


INFO:sagemaker.local.image:===== Job Complete =====


INFO:sagemaker.local.entities:Pipeline step 'TrainCreditModel' SUCCEEDED.


INFO:sagemaker.local.entities:Pipeline execution 99e0af66-8526-4ea2-821b-8253452dd4bf SUCCEEDED


pipeline: Succeeded
  GenerateData Succeeded
  TrainCreditModel Succeeded
